In [19]:
from typing import TypedDict, Optional, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from dotenv import load_dotenv
from datetime import date
from openai import OpenAI
import tempfile

openai_client = OpenAI()

load_dotenv()

True

In [20]:
class GeneralInformation(TypedDict):
    name: Optional[str]
    gender: Optional[str]
    date_of_birth: Optional[str]
    address: Optional[str]
    postal_code: Optional[str]
    email: Optional[str]
    phone: Optional[str]
    has_family_doctor: Optional[bool]
    family_doctor_name: Optional[str]
    has_psychiatrist: Optional[bool]
    psychiatrist_name: Optional[str]

class CurrentConcerns(TypedDict):
    primary_problems: Optional[str]
    goals: Optional[str]

class AliceState(TypedDict): 
    messages: List[dict] 
    current_section: Optional[str] 
    sections_completed: List[str] 
    safety_flag: bool 
    conversation_started: bool 
    
    latest_input_source: str 
    speech_to_text_active: bool 
    general_information: GeneralInformation 
    current_concerns: CurrentConcerns


llm = ChatOpenAI(model="gpt-5.4-mini")



In [21]:
@tool
def save_answer(field_name: str, value: str) -> str:
    """
    Call this every time the patient answers a question or corrects a previous answer.
    field_name: the exact field name (e.g. 'name', 'gender', 'age', 'address')
    value: exactly what the patient said
    """
    return "saved"


In [22]:
@tool
def raise_safety_flag() -> str:
    """
    Call this immediately if the patient mentions:
    - Thoughts of suicide or self-harm
    - Thoughts of harming others
    - A current crisis situation
    """
    return "flag_raised"



In [23]:
tools = [save_answer, raise_safety_flag]
model    = llm.bind_tools(tools)


In [24]:
STATIC_PROMPT = """
You are ALICE, a warm and professional clinical intake assistant
at St. Michael's Hospital, Unity Health Toronto.

Your role:
- Help collect clinical intake information before the patient's appointment.
- Collect information one question at a time.
- Speak calmly, kindly, and professionally.
- Do not diagnose, treat, or give medical advice.
- Do not guess missing information.
- Do not ask for information that has already been collected.
- If an answer is unclear, ask a brief clarification.
- If the patient corrects previous information, save the corrected value.

Conversation rules:
- Ask only one intake question at a time.
- Keep replies short and clear.

Small-talk and orientation rules:
- If the patient seems nervous or confused, reassure them briefly.
- If the patient asks whether they must answer, say they can skip anything they are not comfortable answering.

Tool rules:
- Call save_answer only when the patient gives a clear intake answer.
- Call save_answer when the patient corrects a previous intake answer.
- If the patient gives information for a different intake field, save it to the correct field, then return to the current required question.
- If the patient answers multiple fields in one message, call save_answer separately for each one, then ask for the next pending field.
- Call raise_safety_flag according to the safety protocol below.


Input-source rules:
- The system will tell you whether the patient's latest message was typed
  or produced by speech-to-text.
- Apply the speech-to-text clarification rules below only when:
  "Latest input source: speech-to-text"
- Do not apply these special rules when:
  "Latest input source: typed text"


Speech-to-text name rule — MANDATORY:
- Apply this rule whenever Speech-to-text mode is ON.
- This rule applies to every person's name, including:
  the patient's legal name, family doctor's name, and psychiatrist's name.
- Whenever collecting a person's name through speech-to-text, you MUST ask
  the patient to spell the complete first and last name letter by letter.
- Do this even if the transcription appears clear or the name looks familiar.
- Never assume that a transcribed name is spelled correctly.
- Do not call save_answer for a name until the patient has provided its spelling.
- After receiving the spelling, reconstruct the complete name.
- Save the reconstructed complete name in UPPERCASE.
- Do not ask for spelling when Speech-to-text mode is OFF.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SAFETY PROTOCOL — READ CAREFULLY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

STEP 1 — DETECT DISTRESS SIGNALS

Before every response, ask yourself:
  "Did the patient express any negative emotion, frustration, tension,
   or difficulty — however mild?"

If yes → pause intake and go to Step 2. Always.

You do not need to see an obvious danger sign.
A passing complaint, a frustrated comment about someone, a vague remark
about stress, or a brief expression of sadness is enough to trigger a
check-in. Your job is not to assess severity — it is to notice and ask.
It is always better to check than to assume everything is fine.

STEP 2 — PROBE

Acknowledge what the patient said warmly and briefly.
Then ask one natural question that checks whether they are safe —
directed at themselves, at someone else, or both, depending on what
they said.

Do not use a scripted question. Let the probe follow naturally from
their words. The question should feel like a genuine human concern,
not a clinical checkbox.

Ask the probe ONCE. Do not ask multiple safety questions in a row.

STEP 3 — EVALUATE THE RESPONSE AND DECIDE

After the patient answers your probe, apply this rule:

  RAISE THE FLAG if the response is anything other than a clear,
  confident denial. This includes:
    - Yes, maybe, sometimes, I don't know, possibly, sort of, I guess
    - "I've thought about it", "I'm not sure"
    - Evasion — changing the subject, "I don't want to talk about it",
      going quiet, or deflecting without answering
    - Any answer that leaves doubt

  DO NOT RAISE THE FLAG only if the patient gives a clear, confident no:
    - "No", "No, nothing like that", "I'm fine, just stressed"
    Acknowledge briefly and return to intake.

  WHEN IN DOUBT → RAISE THE FLAG.
  It is always safer to flag than to miss a risk.

STEP 4 — AFTER RAISING THE FLAG
- Call raise_safety_flag immediately.
- Do not write anything else in that turn.
- The system will display the safety message automatically.
- On the next turn, resume intake calmly with the current pending question.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SAFETY EXAMPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

These examples show the pattern. Real conversations will vary — use
your judgment, not just these cases.

Example 1 — Mild frustration about another person → probe → clear no → continue:
  Patient:   "My father is so annoying."
  ALICE:     "That sounds frustrating. Sometimes when things are tense at
              home it can really weigh on you — has it ever gotten to the
              point where you've felt unsafe or had thoughts of hurting
              yourself or someone else?"
  Patient:   "No no, he's just loud."
  Action:    Acknowledge briefly, return to intake. Do NOT flag.

Example 2 — Vague hopelessness → probe → vague answer → flag:
  Patient:   "I just feel like I can't keep going."
  ALICE:     "I'm really sorry you're feeling that way. I want to make
              sure you're safe — have you had any thoughts of hurting
              yourself?"
  Patient:   "I don't know... maybe."
  Action:    call raise_safety_flag()

Example 3 — Anger toward another person → probe → evasive → flag:
  Patient:   "I'm so angry at my brother I don't know what I might do."
  ALICE:     "That sounds like a really difficult situation. I want to make
              sure everyone is safe — have you had any thoughts of hurting
              him or anyone else?"
  Patient:   "I don't really want to talk about that."
  Action:    call raise_safety_flag()

Example 4 — Explicit self-harm or harm to others → skip probe → flag immediately:
  Patient:   "I've been thinking about suicide." / "I want to hurt my roommate."
  Action:    call raise_safety_flag() immediately. No probe needed.

Example 5 — Fear of another person → probe → clear no → continue:
  Patient:   "My partner has a bad temper and it scares me."
  ALICE:     "That sounds really hard. Are you safe right now — is anyone
              hurting or threatening you?"
  Patient:   "No, it's not physical, I'm okay."
  Action:    Acknowledge briefly, return to intake. Do NOT flag.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
IMPORTANT SAFETY CONSTRAINTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Ask the probe ONCE only. Do not repeat or rephrase if the patient deflects.
- Do not ask multiple safety questions back to back.
- Do not provide therapy, advice, or diagnosis.
- Do not try to de-escalate or solve the patient's problem yourself.
- After flagging, do not linger on the topic — resume intake on the next turn.


IMPORTANT EXCEPTION:
Physical health symptoms mentioned during intake are clinical information,
not distress signals. Never trigger the safety probe for physical complaints.

Examples that are NOT distress signals:
- "I have a headache"
- "my back has been hurting"
- "I feel tired all the time"
- "I have chest pain"
- "I haven't been sleeping well"

These are symptoms the patient is reporting as part of their intake.
Treat them as answers to primary_problems, not as safety concerns.

"""



In [25]:
REQUIRED_FIELDS = {
    "general_information": [
        "name",
        "gender",
        "date_of_birth",
        "address",
        "postal_code",
        "email",
        "phone",
        "has_family_doctor",
        "family_doctor_name",
        "has_psychiatrist",
        "psychiatrist_name",
    ],
    "current_concerns": [
        "primary_problems",
        "goals",
    ],
}

In [26]:
import re
from datetime import datetime, date



def validate_date_of_birth(value: str):
    try:
        dob = datetime.strptime(value.strip(), "%d/%m/%Y").date()
        if not (date(1900, 1, 1) <= dob < date.today()):
            return False, "Please enter a valid date of birth."
        return True, dob.strftime("%d/%m/%Y")
    except ValueError:
        return False, "Invalid format. Please use DD/MM/YYYY."


def validate_postal_code(value: str):
    value = value.strip().upper().replace(" ", "")
    if re.match(r'^[A-Z]\d[A-Z]\d[A-Z]\d$', value):
        return True, value
    return False, "Invalid postal code. Please use the format A#A#A#, for example K2C2K9."


def validate_email(value: str):
    value = value.strip()
    if "@" in value and "." in value.split("@")[-1]:
        return True, value
    return False, "Invalid email address."

def validate_address(value: str):
    province_map = {
        "ON": "Ontario",
        "BC": "British Columbia",
        "AB": "Alberta",
        "SK": "Saskatchewan",
        "MB": "Manitoba",
        "QC": "Quebec",
        "NB": "New Brunswick",
        "NS": "Nova Scotia",
        "PE": "Prince Edward Island",
        "NL": "Newfoundland and Labrador",
        "YT": "Yukon",
        "NT": "Northwest Territories",
        "NU": "Nunavut",
    }
    province_names = {v.lower(): v for v in province_map.values()}

    value = value.strip()
    if not value:
        return False, "Please provide a complete address."

    parts = [p.strip() for p in value.split(",")]

    if len(parts) < 2:
        return False, "Please include street, city, and province."

    formatted = []
    for i, part in enumerate(parts):
        if i == len(parts) - 1:
            key_upper = part.strip().upper()
            key_lower = part.strip().lower()
            if key_upper in province_map:
                formatted.append(province_map[key_upper])
            elif key_lower in province_names:
                formatted.append(province_names[key_lower])
            else:
                formatted.append(part.strip().title())
        else:
            formatted.append(part.strip().title())

    return True, ", ".join(formatted)

def validate_phone(value: str):
    digits = re.sub(r'\D', '', value)
    if 10 <= len(digits) <= 15:
        return True, digits
    return False, "Phone number must be 10 digits (or include country code)."

def validate_boolean(value: str):
    v = value.lower().strip()
    if v.startswith(("yes", "y", "true")):
        return True, True
    if v.startswith(("no", "n", "false")):
        return True, False
    return False, "Please answer yes or no."

def validate_doctor_name(value: str):
    if value.strip().lower() == "unknown":
        return True, "unknown"
    return True, value.strip().upper()


FIELD_VALIDATORS = {
    "date_of_birth":      validate_date_of_birth,
    "postal_code":        validate_postal_code,
    "email":              validate_email,
    "phone":              validate_phone,
    "has_family_doctor":  validate_boolean,
    "has_psychiatrist":   validate_boolean,
    "family_doctor_name": validate_doctor_name,
    "psychiatrist_name":  validate_doctor_name,
    "validate_address" :validate_address,
}

In [27]:

SECTION_PROMPTS = {
    "general_information": """
Collect the following fields in order:
name, gender, date_of_birth, address, postal_code, email, phone,
has_family_doctor, family_doctor_name, has_psychiatrist, psychiatrist_name

Speech-to-text rule for this section:
- Apply the speech-specific instructions below only when:
  Speech-to-text mode: ON
- When speech-to-text mode is OFF, follow the normal field rules.
- Ask for spelling only when exact spelling is uncertain.
- Do not guess uncertain names, addresses, or email addresses.
- Ask about only one unclear part at a time.
- Do not save a field until the necessary clarification is complete.

Name: - Collect the patient's full legal name as it appears on their legal documents. 
- If Speech-to-text mode is ON, ask the patient to say their full legal name and spell their complete first and last name letter by letter. 
- Spelling is mandatory when Speech-to-text mode is ON, even when the transcribed name appears clear. 
- Do not save the name before receiving the spelling. 
- Reconstruct the complete legal name from the provided spelling. 
- Save the reconstructed name in UPPERCASE. 
- If the patient gives only one name, ask: "Is that your complete legal name?" 
- If they confirm that it is complete, still ask them to spell it when Speech-to-text mode is ON.


Gender:
- Ask how the patient would like their gender recorded.
- Do not assume gender from their name.
- Save the complete answer with the first letter uppercase,
  exactly as the patient states it.
- Do not normally ask the patient to spell their gender response.

Date of birth:
- Ask the patient for their date of birth in DD/MM/YYYY format.
- If the patient gives it in DD/MM/YYYY already, save it as is.
- If the patient gives it in a written format such as
  "March 4th 1990" or "4 March 1990",
  convert it to DD/MM/YYYY before saving.
- Example:
  "March 4th 1990" → save_answer("date_of_birth", "04/03/1990")
- Example:
  "4 March 1990" → save_answer("date_of_birth", "04/03/1990")
- Example:
  "04/03/1990" → save_answer("date_of_birth", "04/03/1990")
- If the date is unclear, ask the patient to repeat it in DD/MM/YYYY format.
- When speech-to-text mode is ON, confirm unclear numbers rather than
  asking the patient to spell them.

Address:
- Collect the patient's complete legal home address.
- It must include street address, city, and province.
- Do not save it until all three parts have been collected.
- Save it in this format:
  [Street Address], [City], [Province]
- Example:
  123 Main Street, Toronto, Ontario
- When speech-to-text mode is ON and the street name or city spelling
  may be uncertain, ask the patient to spell only the uncertain part.
- Do not ask the patient to spell common address words such as
  Street, Road, Avenue, Drive, Boulevard, Lane, or Court unless necessary.
- Reconstruct the complete address after clarification.
- Confirm the complete address before saving it.

Postal code:
- Collect the patient's Canadian postal code.
- If it looks incorrect, ask the patient to repeat it.
- When speech-to-text mode is ON, ask the patient to say each letter
  and number separately if the postal code is unclear.
- Save it without spaces.

Email:
- Collect the patient's email address.
- If it does not look valid, ask them to repeat it.
- When speech-to-text mode is ON, confirm unclear letters, numbers,
  symbols, and domain names before saving it.
- Ask the patient to spell the email address only when needed.
- Do not guess between similar-sounding letters or words.

Phone:
- Collect the patient's phone number.
- If it seems too short or unclear, ask them to repeat it.
- When speech-to-text mode is ON, ask the patient to repeat the digits
  individually if any number is unclear.

Family doctor: - Ask whether the patient currently has a family doctor. 
- Save the answer as "yes" or "no". 
- If yes, ask for the doctor's complete first and last name. 
- If Speech-to-text mode is ON, you MUST ask the patient to spell the doctor's complete first and last name letter by letter. 
- Do not save the doctor's name before receiving the spelling. 
- Reconstruct the complete name and save it in UPPERCASE. 
- If the patient does not know the name, save "unknown". 
- If no, skip the family doctor name.

Psychiatrist: - Ask whether the patient currently has a psychiatrist. 
- Save the answer as "yes" or "no". 
- If yes, ask for the psychiatrist's complete first and last name. 
- If Speech-to-text mode is ON, you MUST ask the patient to spell the psychiatrist's complete first and last name letter by letter. 
- Do not save the psychiatrist's name before receiving the spelling. 
- Reconstruct the complete name and save it in UPPERCASE. 
- If the patient does not know the name, save "unknown". 
- If no, skip the psychiatrist name.
""",


    "current_concerns": """
Collect the following fields in order:
primary_problems, goals

General rules:
- Let the patient answer in their own words.
- Do not diagnose, interpret, or provide therapy.
- If an answer is unclear, ask one short clarification.
- Always call save_answer as soon as the patient gives a clear response.
- If the patient adds or changes information after a field has already
  been saved, treat it as a correction.
- Fix obvious transcription, spelling, and sentence-structure errors.
- Preserve the patient's intended meaning exactly.
- Do not ask the patient to spell ordinary words in their description.
- Ask for clarification only when a transcription error changes or obscures
  the patient's meaning.
- After a correction, call save_answer with the updated value.
- Do not re-ask the original question after a correction.

Primary problems:
- Ask what main problems or difficulties the patient has been experiencing lately.
- If the patient reports no problems, call:
  save_answer(
      "primary_problems",
      "No primary concerns reported at this time."
  )
- If the patient describes their problems, improve spelling and sentence
  structure before saving.
- Keep the patient's meaning exactly as intended.

Goals:
- Ask what the patient hopes to get out of their appointment.
- If the patient reports no specific goals, call:
  save_answer(
      "goals",
      "No specific goals identified for this appointment."
  )
- If the patient describes their goals, improve spelling and sentence
  structure before saving.
- Keep the patient's meaning exactly as intended.
""",
}



In [28]:
def get_pending_fields(state: AliceState, section: str) -> list:
    section_data = state[section]
    return [
        f for f in REQUIRED_FIELDS[section]
        if section_data.get(f) is None
        and not (f == "family_doctor_name" and section_data.get("has_family_doctor") is False)
        and not (f == "psychiatrist_name" and section_data.get("has_psychiatrist") is False)
    ]

In [29]:
def build_section_prompt(state: AliceState) -> str:
    today        = date.today().strftime("%B %d, %Y")
    name         = state["general_information"].get("name")
    section      = state["current_section"]
    pending      = get_pending_fields(state, section)
    next_field   = pending[0] if pending else None

    if not state["conversation_started"]:
        greeting = "This is the start. Greet the patient and introduce yourself as ALICE."
    else:
        greeting = "You have already introduced yourself. Do not greet again."

    return (
        f"Today's date: {today}\n"
        f"Patient name: {name if name else 'not collected yet'}\n"
        f"{greeting}\n\n"
        f"Next field to collect: {next_field}\n"
        f"All pending fields: {pending}\n\n"
        f"{SECTION_PROMPTS[section]}\n\n"
        "When all fields are collected say: "
        "'All done! Please take a look at the panel on the right — "
        "if anything looks incorrect or if you'd like to add or change anything, just let me know.'"
    )

In [30]:
def build_system_prompt(state: AliceState) -> str:
    input_source = state.get(
        "latest_input_source",
        "typed text"
    )

    speech_mode = (
        "ON"
        if state.get("speech_to_text_active", False)
        else "OFF"
    )

    voice_context = (
        f"Speech-to-text mode: {speech_mode}\n"
        f"Latest input source: {input_source}"
    )

    return (
        STATIC_PROMPT
        + "\n\n"
        + voice_context
        + "\n\n"
        + build_section_prompt(state)
    )

In [31]:
import json
import os
from datetime import datetime

def save_record(state: AliceState):
    os.makedirs("records", exist_ok=True)

    name     = state["general_information"].get("name", "unknown")
    dob      = state["general_information"].get("date_of_birth", "unknown")
    name_slug = name.replace(" ", "_").lower()
    dob_slug  = dob.replace("/", "-")
    filename  = f"records/{name_slug}_{dob_slug}.json"

    record = {
        "saved_at":            datetime.now().isoformat(),
        "general_information": state["general_information"],
        "current_concerns":    state["current_concerns"],
        "safety_flag":         state["safety_flag"],
    }

    with open(filename, "w") as f:
        json.dump(record, f, indent=2, default=str)

In [32]:
def section_node(state: AliceState) -> AliceState:
    section = state["current_section"]
    section_data = state[section]                  
    model   = ChatOpenAI(model="gpt-5.4-mini").bind_tools(tools)
    system  = {"role": "system", "content": build_system_prompt(state)}

    response = model.invoke([system] + state["messages"])

    safety_raised = False
    tool_results  = {}

    for tc in response.tool_calls:
        if tc["name"] == "save_answer":
            field     = tc["args"]["field_name"]
            value     = tc["args"]["value"]
            validator = FIELD_VALIDATORS.get(field)

            if validator:
                valid, result = validator(value)
                if valid:
                    state[section][field]  = result
                    tool_results[tc["id"]] = "saved"
                else:
                    tool_results[tc["id"]] = f"invalid: {result}"
            else:
                state[section][field]      = value
                tool_results[tc["id"]]     = "saved"

        elif tc["name"] == "raise_safety_flag":
            state["safety_flag"]           = True
            safety_raised                  = True
            tool_results[tc["id"]]         = "flag_raised"


    if not response.tool_calls:
        
        reply         = response.content
        tool_exchange = []

    elif safety_raised:
        
        state["messages"][-1] = {"role": "user", "content": "safety flag was raised and handled."}
        reply = (
            "I'm really glad you shared that with me. "
            "You are not alone. Please call 988 or go to your nearest emergency room. "
            "A member of our care team has been notified. "
            "Whenever you're ready, we can continue with the intake."
        )
        tool_exchange = []

    else:
        
        tool_exchange = [response] + [
            {"role": "tool", "content": tool_results[tc["id"]], "tool_call_id": tc["id"]}
            for tc in response.tool_calls
        ]
        updated_system = {"role": "system", "content": build_system_prompt(state)}
        reply = llm.invoke([updated_system] + state["messages"] + tool_exchange).content
        
    pending = get_pending_fields(state, section)
    if not pending and section not in state["sections_completed"]:
        state["sections_completed"].append(section)
        
    
    if set(state["sections_completed"]) == {"general_information", "current_concerns"}:
        save_record(state)


    state["messages"]             = state["messages"] + tool_exchange + [{"role": "assistant", "content": reply}]
    state["conversation_started"] = True
    return state

In [33]:
builder = StateGraph(AliceState)
builder.add_node("section", section_node)
builder.set_entry_point("section")
builder.add_edge("section", END)

graph = builder.compile()


In [34]:

def text_to_speech(text):
    """Convert an ALICE text response into a temporary MP3 file."""
    if not text or not str(text).strip():
        return None

    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp3",
    )
    temp_file.close()

    try:
        response = openai_client.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="shimmer",
            input=str(text),
            instructions=(
                "Speak with a warm, friendly, young adult feminine voice. "
                "Sound calm, caring, natural, and professional. "
                "Use a comfortable and clear speaking pace."
            ),
        )

        response.write_to_file(temp_file.name)
        return temp_file.name

    except Exception as error:
        print(f"Text-to-speech error: {error}")

        try:
            os.remove(temp_file.name)
        except OSError:
            pass

        return None



def transcribe_audio(audio_file):
    if not audio_file:
        return None

    try:
        with open(audio_file, "rb") as audio:
            transcription = openai_client.audio.transcriptions.create(
                model='gpt-4o-transcribe',
                file=audio,
                prompt=(
                    "This is a Canadian clinical intake conversation. "
                    "The speaker may provide personal names, doctor names, "
                    "street names, city names, postal codes, dates, phone "
                    "numbers, and email addresses. The speaker may spell "
                    "names letter by letter. Preserve every spelled letter "
                    "carefully and do not combine or correct the letters."
                )
            )

        return transcription.text.strip()

    except Exception as error:
        print(f"Speech-to-text error: {error}")
        return None

In [35]:
initial_state: AliceState = {
    "messages": [],
    "current_section": None,
    "sections_completed": [],
    "safety_flag": False,
    "conversation_started": False,
    "latest_input_source": "typed text", 
    "speech_to_text_active": False,
    "general_information": {
        "name": None,
        "gender": None,
        "date_of_birth": None,
        "postal_code":None,
        "address": None,
        "email": None,
        "phone": None,
        "has_family_doctor": None,
        "family_doctor_name": None,
        "has_psychiatrist": None,
        "psychiatrist_name": None,
    },   
    "current_concerns": {
        "primary_problems": None,
        "goals": None,
    },
}



In [ ]:
import copy
import html
import os
import tempfile

import gradio as gr
from openai import OpenAI
from PIL import Image, ImageDraw



def get_latest_alice_message(history):
    """Return the latest assistant text currently displayed in the chat."""
    for message in reversed(history or []):
        if (
            isinstance(message, dict)
            and message.get("role") == "assistant"
        ):
            content = message.get("content")

            if isinstance(content, str):
                return content

    return None


def toggle_alice_voice(currently_enabled, history):
    """
    Toggle ALICE text-to-speech.

    ALICE voice starts ON. When it is turned back on,
    the latest ALICE message is spoken immediately.
    """
    new_state = not currently_enabled

    if new_state:
        latest_message = get_latest_alice_message(history)
        audio_file = text_to_speech(latest_message)

        return (
            new_state,
            gr.update(value="🔊 ALICE Voice: ON"),
            audio_file,
        )

    return (
        new_state,
        gr.update(value="🔇 ALICE Voice: OFF"),
        None,
    )


def toggle_patient_voice(currently_enabled, alice_state):
    """
    Toggle patient speech-to-text and store the mode in AliceState.

    Patient Voice starts OFF.
    """
    if alice_state is None:
        alice_state = get_fresh_state()

    new_state = not currently_enabled
    alice_state["speech_to_text_active"] = new_state
    alice_state["latest_input_source"] = "typed text"

    if new_state:
        return (
            new_state,
            alice_state,
            gr.update(value="🎤 Patient Voice: ON"),
            gr.update(visible=True, value=None),
            "Patient Voice is on. Record an answer; it will be sent automatically.",
        )

    return (
        new_state,
        alice_state,
        gr.update(value="🎤 Patient Voice: OFF"),
        gr.update(visible=False, value=None),
        "Patient Voice is off.",
    )

# =========================================================
# AVATARS
# =========================================================

def make_avatar(color, size=80):
    image = Image.new(
        "RGBA",
        (size, size),
        (0, 0, 0, 0),
    )

    draw = ImageDraw.Draw(image)

    draw.ellipse(
        [0, 0, size - 1, size - 1],
        fill=color,
    )

    path = os.path.join(
        tempfile.gettempdir(),
        f"avatar_{color[0]}.png",
    )

    image.save(path)
    return path


ALICE_AVATAR = "https://img.icons8.com/color/96/nurse-female.png"
USER_AVATAR = make_avatar((99, 115, 129))


# =========================================================
# INFORMATION PANEL
# =========================================================

FIELD_LABELS = {
    "general_information": {
        "name": "Full Name",
        "gender": "Gender",
        "date_of_birth": "Date of Birth",
        "address": "Address",
        "postal_code": "Postal Code",
        "email": "Email",
        "phone": "Phone",
        "has_family_doctor": "Has Family Doctor",
        "family_doctor_name": "Family Doctor Name",
        "has_psychiatrist": "Has Psychiatrist",
        "psychiatrist_name": "Psychiatrist Name",
    },
    "current_concerns": {
        "primary_problems": "Primary Problems",
        "goals": "Goals",
    },
}


def format_display(alice_state):
    if not alice_state or not alice_state.get("current_section"):
        return """
        <div class="empty-info">
            Select a section above to begin
        </div>
        """

    section = alice_state["current_section"]
    data = alice_state[section]
    labels = FIELD_LABELS.get(section, {})
    title = section.replace("_", " ").title()

    def should_show(field):
        if (
            field == "family_doctor_name"
            and data.get("has_family_doctor") is False
        ):
            return False

        if (
            field == "psychiatrist_name"
            and data.get("has_psychiatrist") is False
        ):
            return False

        return True

    visible_fields = {
        field: label
        for field, label in labels.items()
        if should_show(field)
    }

    total = len(visible_fields)

    filled = sum(
        1
        for field in visible_fields
        if data.get(field) is not None
    )

    percentage = int((filled / total) * 100) if total else 0
    fields_html = ""

    for field, label in visible_fields.items():
        value = data.get(field)
        safe_label = html.escape(str(label))

        if value is None:
            value_html = '<div class="field-empty">—</div>'

        elif isinstance(value, bool):
            badge_class = "yes-badge" if value else "no-badge"
            badge_text = "Yes" if value else "No"

            value_html = (
                f'<span class="boolean-badge {badge_class}">'
                f'{badge_text}</span>'
            )

        else:
            safe_value = html.escape(str(value))
            value_html = (
                f'<div class="field-value">{safe_value}</div>'
            )

        fields_html += f"""
        <div class="info-field">
            <div class="field-label">{safe_label}</div>
            {value_html}
        </div>
        """

    safe_title = html.escape(title)

    return f"""
    <div class="information-panel">
        <div class="information-header">
            <div class="information-title">{safe_title}</div>
            <div class="information-count">{filled}/{total}</div>
        </div>

        <div class="progress-track">
            <div
                class="progress-value"
                style="width:{percentage}%;">
            </div>
        </div>

        {fields_html}
    </div>
    """


# =========================================================
# CSS
# =========================================================

custom_css = """
@import url(
    'https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap'
);

*, body, .gradio-container {
    font-family:
        'Inter',
        -apple-system,
        BlinkMacSystemFont,
        sans-serif !important;
}

body,
.gradio-container {
    background: #dce8f5 !important;
}

.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto !important;
    padding: 24px !important;
}

.alice-header {
    display: flex;
    align-items: center;
    gap: 16px;
    background: white;
    border: 1px solid #c5d8f0;
    border-radius: 14px;
    padding: 20px 26px;
    margin-bottom: 16px;
}

.alice-header h1 {
    color: #1e293b !important;
    font-size: 22px !important;
    font-weight: 600 !important;
    margin: 0 !important;
    padding: 0 !important;
}

.alice-header p {
    color: #64748b !important;
    font-size: 13px !important;
    margin: 3px 0 0 !important;
}

.btn-general button,
.btn-concerns button {
    width: 100% !important;
    padding: 14px 20px !important;
    border-radius: 10px !important;
    background: white !important;
    color: #1e293b !important;
    font-size: 14px !important;
    font-weight: 500 !important;
    text-align: left !important;
    box-shadow: none !important;
    transition: background 0.15s ease !important;
}

.btn-general button {
    border: 1px solid #c5d8f0 !important;
    border-left: 3px solid #185FA5 !important;
}

.btn-concerns button {
    border: 1px solid #c5d8f0 !important;
    border-left: 3px solid #0F6E56 !important;
}

.btn-general button:hover,
.btn-concerns button:hover {
    background: #f0f6ff !important;
}

.voice-card {
    min-height: 132px;
    background: white !important;
    border: 1px solid #c5d8f0 !important;
    border-radius: 12px !important;
    padding: 14px !important;
}

.voice-title {
    margin-bottom: 8px;
    color: #334155;
    font-size: 13px;
    font-weight: 600;
}

.voice-description {
    min-height: 34px;
    color: #64748b;
    font-size: 12px;
    line-height: 1.45;
}

.alice-voice-button button {
    border: 1px solid #93c5fd !important;
    background: #eff6ff !important;
    color: #1d4ed8 !important;
}

.patient-voice-button button {
    border: 1px solid #86efac !important;
    background: #f0fdf4 !important;
    color: #166534 !important;
}

/*
The ALICE audio element remains rendered so autoplay can work,
but the player controls are not shown.
*/
#alice-audio-output {
    display: none !important;
}

.patient-recorder {
    margin-top: 8px !important;
}

.patient-status {
    color: #64748b;
    font-size: 12px;
}

.ai-disclosure {
    color: #64748b;
    font-size: 11px;
    margin-top: 4px;
}

.info-box {
    background: #f0f6ff !important;
    border: 1px solid #c5d8f0 !important;
    border-radius: 14px !important;
    padding: 18px !important;
}

.empty-info {
    padding: 20px;
    color: #94a3b8;
    font-size: 13px;
    font-style: italic;
    text-align: center;
}

.information-panel {
    padding: 4px 8px;
}

.information-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 12px;
    padding-bottom: 12px;
    border-bottom: 2px solid #dbeafe;
}

.information-title {
    color: #1e40af;
    font-size: 13px;
    font-weight: 600;
}

.information-count {
    color: #94a3b8;
    font-size: 11px;
}

.progress-track {
    height: 5px;
    margin-bottom: 16px;
    overflow: hidden;
    border-radius: 999px;
    background: #e2e8f0;
}

.progress-value {
    height: 5px;
    border-radius: 999px;
    background: linear-gradient(90deg, #185FA5, #3b82f6);
}

.info-field {
    padding: 11px 0;
    border-bottom: 1px solid #e8f0fb;
}

.field-label {
    margin-bottom: 5px;
    color: #94a3b8;
    font-size: 10px;
    letter-spacing: 0.6px;
    text-transform: uppercase;
}

.field-value {
    color: #1e293b;
    font-size: 14px;
    font-weight: 500;
    overflow-wrap: anywhere;
}

.field-empty {
    color: #cbd5e0;
    font-size: 13px;
    font-style: italic;
}

.boolean-badge {
    display: inline-block;
    padding: 2px 10px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 600;
}

.yes-badge {
    color: #0F6E56;
    background: #e6f4ef;
}

.no-badge {
    color: #718096;
    background: #f1f5f9;
}

@media (max-width: 760px) {
    .gradio-container {
        padding: 12px !important;
    }

    .alice-header {
        padding: 16px 18px;
    }
}
"""


# =========================================================
# STATE AND CHAT CALLBACKS
# =========================================================


# =========================================================
# STATE AND CHAT CALLBACKS
# =========================================================

def get_fresh_state():
    state = copy.deepcopy(initial_state)

    # These defaults prevent errors if the fields have not yet
    # been added manually to initial_state.
    state.setdefault("latest_input_source", "typed text")
    state.setdefault("speech_to_text_active", False)

    return state


def start():
    """Initialize the UI. ALICE voice starts enabled."""
    state = get_fresh_state()

    welcome_text = (
        "Welcome! Please select a section above "
        "to begin your intake."
    )

    welcome_history = [{
        "role": "assistant",
        "content": f"👋 {welcome_text}",
    }]

    # A browser may block this first autoplay until the first interaction.
    welcome_audio = text_to_speech(welcome_text)

    return (
        welcome_history,
        state,
        format_display(state),
        welcome_audio,
    )


def run_patient_message(
    message,
    displayed_user_message,
    history,
    alice_state,
    alice_voice_enabled,
    input_source,
    speech_to_text_active,
):
    """Shared logic for typed and spoken patient messages."""
    if alice_state is None:
        alice_state = get_fresh_state()

    history = history or []

    if alice_state.get("current_section") is None:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
        )

    if not message or not message.strip():
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
        )

    alice_state["latest_input_source"] = input_source
    alice_state["speech_to_text_active"] = speech_to_text_active

    alice_state["messages"] = (
        alice_state.get("messages", [])
        + [{
            "role": "user",
            "content": message.strip(),
        }]
    )

    alice_state = graph.invoke(alice_state)
    reply = alice_state["messages"][-1]["content"]

    updated_history = history + [
        {
            "role": "user",
            "content": displayed_user_message,
        },
        {
            "role": "assistant",
            "content": reply,
        },
    ]

    reply_audio = (
        text_to_speech(reply)
        if alice_voice_enabled
        else None
    )

    return (
        updated_history,
        alice_state,
        format_display(alice_state),
        reply_audio,
    )


def chat(
    message,
    history,
    alice_state,
    alice_voice_enabled,
    patient_voice_enabled,
):
    """Submit a typed patient message to LangGraph."""
    clean_message = message.strip() if message else ""

    return run_patient_message(
        message=clean_message,
        displayed_user_message=clean_message,
        history=history,
        alice_state=alice_state,
        alice_voice_enabled=alice_voice_enabled,
        input_source="typed text",
        speech_to_text_active=patient_voice_enabled,
    )


def voice_chat(
    audio_file,
    history,
    alice_state,
    alice_voice_enabled,
    patient_voice_enabled,
):
    """
    Transcribe a recording privately and submit it immediately.

    The transcript is used internally by LangGraph, but the UI displays
    only a voice-response marker instead of exposing the transcript.
    """
    if alice_state is None:
        alice_state = get_fresh_state()

    history = history or []

    if not patient_voice_enabled:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
            gr.update(value=None),
            "Patient Voice is off.",
        )

    if alice_state.get("current_section") is None:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
            gr.update(value=None),
            "Please select an intake section before recording.",
        )

    transcript = transcribe_audio(audio_file)

    if not transcript:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
            gr.update(value=None),
            "I could not understand the recording. Please try again.",
        )

    (
        updated_history,
        updated_state,
        updated_display,
        reply_audio,
    ) = run_patient_message(
        message=transcript,
        displayed_user_message="🎤 Voice response",
        history=history,
        alice_state=alice_state,
        alice_voice_enabled=alice_voice_enabled,
        input_source="speech-to-text",
        speech_to_text_active=True,
    )

    return (
        updated_history,
        updated_state,
        updated_display,
        reply_audio,
        gr.update(value=None),
        "Voice response sent automatically.",
    )


def switch_section(
    section,
    alice_state,
    alice_voice_enabled,
    patient_voice_enabled,
):
    """Open an intake section and ask its first pending question."""
    if alice_state is None:
        alice_state = get_fresh_state()

    alice_state["current_section"] = section
    alice_state["messages"] = []
    alice_state["latest_input_source"] = "typed text"
    alice_state["speech_to_text_active"] = patient_voice_enabled

    alice_state = graph.invoke(alice_state)
    reply = alice_state["messages"][-1]["content"]

    section_history = [{
        "role": "assistant",
        "content": reply,
    }]

    reply_audio = (
        text_to_speech(reply)
        if alice_voice_enabled
        else None
    )

    return (
        section_history,
        alice_state,
        format_display(alice_state),
        reply_audio,
    )

# =========================================================
# GRADIO INTERFACE
# =========================================================

with gr.Blocks(
    title="ALICE",
    css=custom_css,
    theme=gr.themes.Soft(),
) as demo:

    alice_state = gr.State(None)

    # ALICE text-to-speech is ON by default.
    alice_voice_enabled = gr.State(True)

    # Patient speech-to-text is OFF by default.
    patient_voice_enabled = gr.State(False)

    gr.HTML("""
        <div class="alice-header">
            <div style="
                width:52px;
                height:52px;
                border-radius:50%;
                background:#dbeafe;
                display:flex;
                align-items:center;
                justify-content:center;
                flex-shrink:0;
                font-size:26px;
            ">
                👩‍⚕️
            </div>

            <div>
                <h1>ALICE</h1>
                <p>Clinical Intake Assistant</p>
            </div>
        </div>
    """)

    with gr.Row():
        btn_general = gr.Button(
            "General Information",
            elem_classes="btn-general",
        )

        btn_concerns = gr.Button(
            "Current Concerns",
            elem_classes="btn-concerns",
        )

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                height=520,
                show_label=False,
                avatar_images=(
                    USER_AVATAR,
                    ALICE_AVATAR,
                ),
            )

            msg = gr.Textbox(
                placeholder="Type your message here...",
                show_label=False,
                container=False,
            )

            with gr.Row():
                with gr.Column(elem_classes="voice-card"):
                    gr.HTML("""
                        <div class="voice-title">ALICE Voice</div>
                        <div class="voice-description">
                            Reads ALICE's replies aloud automatically.
                        </div>
                    """)

                    alice_voice_button = gr.Button(
                        "🔊 ALICE Voice: ON",
                        elem_classes="alice-voice-button",
                    )

                with gr.Column(elem_classes="voice-card"):
                    gr.HTML("""
                        <div class="voice-title">Patient Voice</div>
                        <div class="voice-description">
                            Records and sends the patient's answer automatically.
                        </div>
                    """)

                    patient_voice_button = gr.Button(
                        "🎤 Patient Voice: OFF",
                        elem_classes="patient-voice-button",
                    )

            alice_audio_output = gr.Audio(
                autoplay=True,
                interactive=False,
                show_label=False,
                container=False,
                elem_id="alice-audio-output",
            )

            # Hidden until Patient Voice is enabled.
            patient_audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Record your answer",
                visible=False,
                elem_classes="patient-recorder",
            )

            patient_status = gr.Markdown(
                "Patient Voice is off.",
                elem_classes="patient-status",
            )

            gr.Markdown(
                (
                    "ALICE's voice and patient transcription are AI-generated. "
                    "Patient recordings are transcribed and submitted automatically."
                ),
                elem_classes="ai-disclosure",
            )

        with gr.Column(scale=1):
            info_display = gr.HTML(
                value=format_display(None),
                elem_classes="info-box",
                label="Collected Information",
            )



    alice_voice_button.click(
        toggle_alice_voice,
        inputs=[
            alice_voice_enabled,
            chatbot,
        ],
        outputs=[
            alice_voice_enabled,
            alice_voice_button,
            alice_audio_output,
        ],
    )

    patient_voice_button.click(
        toggle_patient_voice,
        inputs=[
            patient_voice_enabled,
            alice_state,
        ],
        outputs=[
            patient_voice_enabled,
            alice_state,
            patient_voice_button,
            patient_audio_input,
            patient_status,
        ],
    )

    patient_audio_input.stop_recording(
        voice_chat,
        inputs=[
            patient_audio_input,
            chatbot,
            alice_state,
            alice_voice_enabled,
            patient_voice_enabled,
        ],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
            patient_audio_input,
            patient_status,
        ],
    )



    msg.submit(
        chat,
        inputs=[
            msg,
            chatbot,
            alice_state,
            alice_voice_enabled,
            patient_voice_enabled,
        ],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
        ],
    ).then(
        lambda: "",
        outputs=msg,
    )

    btn_general.click(
        lambda state, alice_voice, patient_voice: switch_section(
            "general_information",
            state,
            alice_voice,
            patient_voice,
        ),
        inputs=[
            alice_state,
            alice_voice_enabled,
            patient_voice_enabled,
        ],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
        ],
    )

    btn_concerns.click(
        lambda state, alice_voice, patient_voice: switch_section(
            "current_concerns",
            state,
            alice_voice,
            patient_voice,
        ),
        inputs=[
            alice_state,
            alice_voice_enabled,
            patient_voice_enabled,
        ],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
        ],
    )

    demo.load(
        start,
        inputs=[],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
        ],
    )


demo.launch()


/var/folders/v9/yz64p8z5579c6d6yx28446qh0000gn/T/ipykernel_30057/1548919434.py:733: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
